<a href="https://colab.research.google.com/github/Chandu1722/ML/blob/main/Kaggle/Accident2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd

In [21]:
train=pd.read_csv('train.csv')
train

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents,accident_risk
0,0,urban,2,0.06,35,daylight,rainy,False,True,afternoon,False,True,1,0.13
1,1,urban,4,0.99,35,daylight,clear,True,False,evening,True,True,0,0.35
2,2,rural,4,0.63,70,dim,clear,False,True,morning,True,False,2,0.30
3,3,highway,4,0.07,35,dim,rainy,True,True,morning,False,False,1,0.21
4,4,rural,1,0.58,60,daylight,foggy,False,False,evening,True,False,1,0.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
517749,517749,highway,4,0.10,70,daylight,foggy,True,True,afternoon,False,False,2,0.32
517750,517750,rural,4,0.47,35,daylight,rainy,True,True,morning,False,False,1,0.26
517751,517751,urban,4,0.62,25,daylight,foggy,False,False,afternoon,False,True,0,0.19
517752,517752,highway,3,0.63,25,night,clear,True,False,afternoon,True,True,3,0.51


In [22]:
test=pd.read_csv('test.csv')
test

,id,road_type,num_lanes,curvature,speed_limit,lighting,weather,road_signs_present,public_road,time_of_day,holiday,school_season,num_reported_accidents
0,517754,highway,2,0.34,45,night,clear,True,True,afternoon,True,True,1
1,517755,urban,3,0.04,45,dim,foggy,True,False,afternoon,True,False,0
2,517756,urban,2,0.59,35,dim,clear,True,False,afternoon,True,True,1
3,517757,rural,4,0.95,35,daylight,rainy,False,False,afternoon,False,False,2
4,517758,highway,2,0.86,35,daylight,clear,True,False,evening,False,True,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...
172580,690334,rural,2,0.01,45,dim,rainy,False,False,afternoon,True,True,2
172581,690335,rural,1,0.74,70,daylight,foggy,False,True,afternoon,False,False,2
172582,690336,urban,2,0.14,70,dim,clear,False,False,evening,True,True,1
172583,690337,urban,1,0.09,45,daylight,foggy,True,True,morning,False,True,0


In [33]:
train['weather'].unique()

array(['rainy', 'clear', 'foggy'], dtype=object)

In [48]:
nominal_cols=['road_type','lighting','weather','time_of_day']
numeric_cols=['num_lanes','curvature','num_reported_accidents','speed_limit']
ordinal_cols=['road_signs_present','public_road','holiday','school_season']
seq_order=[[False,True],[False,True],[False,True],[False,True]]

In [49]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
Transformer=ColumnTransformer([('Nominal',OneHotEncoder(),nominal_cols),
 ('Ordinal',OrdinalEncoder(categories=seq_order),ordinal_cols),
  ('Numeric',StandardScaler(),numeric_cols)])

In [50]:
X=train.drop(columns=['accident_risk','id'])
y=train['accident_risk']

In [51]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [41]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
model=Pipeline([('Transformer',Transformer),('Model',RandomForestRegressor())])
model.fit(X_train,y_train)
y_pred_rf=model.predict(X_test)
msq=mean_squared_error(y_test,y_pred_rf)
print(msq)

0.003529718637475965


In [42]:
print(msq**0.5)

0.05941143524167688


In [43]:
pred=model.predict(test.drop(columns=['id']))
output=pd.DataFrame({'id':test['id'],'accident_risk':pred})
output.to_csv('submission3.csv', index=False)

In [52]:
from lightgbm import LGBMRegressor
model_l=Pipeline([('Transformer',Transformer),('Model',LGBMRegressor())])
model_l.fit(X_train,y_train)
y_pred_l=model_l.predict(X_test)
msq_l=mean_squared_error(y_test,y_pred_l)
print(msq_l**0.5)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018126 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 209
[LightGBM] [Info] Number of data points in the train set: 414203, number of used features: 20
[LightGBM] [Info] Start training from score 0.352605


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


0.056342202858803404


In [54]:
from xgboost import XGBRegressor
model_x=Pipeline([('Transformer',Transformer),('Model',XGBRegressor())])
model_x.fit(X_train,y_train)
y_pred_x=model_x.predict(X_test)
msq_x=mean_squared_error(y_test,y_pred_x)
print(msq_x**0.5)

0.05627165011919581


In [47]:
pred=model_x.predict(test.drop(columns=['id']))
output=pd.DataFrame({'id':test['id'],'accident_risk':pred})
output.to_csv('submission5.csv', index=False)